# Logistic Regression tuning

Goal:
Tune Logistic Regression using GridSearchCV on the training data only.

We will tune:
- C
- penalty
- class_weight

The final X_test set is not used during tuning.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)

Current working directory: c:\temp\python_learning\ml_projects\diabetes_predictions\notebooks
Project root: c:\temp\python_learning\ml_projects\diabetes_predictions


In [2]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

from src.data_utils import load_data, split_features_target, make_train_test_split
from src.evaluation import make_stratified_cv, get_classification_scoring
from src.pipelines import build_classification_pipeline

In [3]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "diabetes.csv"

df = load_data(DATA_PATH)
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = make_train_test_split(X, y)

cv = make_stratified_cv()
scoring = get_classification_scoring()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

X_train shape: (614, 8)
X_test shape: (154, 8)

Train target distribution:
Outcome
0    0.651466
1    0.348534
Name: proportion, dtype: float64

Test target distribution:
Outcome
0    0.649351
1    0.350649
Name: proportion, dtype: float64


In [11]:
base_pipeline = build_classification_pipeline(
    model=LogisticRegression(
        max_iter=2000,
        random_state=42,
    ),
    use_scaler=True,
    use_indicators=True,
)

base_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('zero_handler', ...), ('imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function rep...001ACCC2E4BF0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments t

In [12]:
param_grid = {
    "model__C": [0.1, 0.3, 1, 3, 10, 30],
    "model__l1_ratio": [0],
    "model__class_weight": ["balanced"],
}

In [13]:
grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.1, 0.3, ...], 'model__class_weight': ['balanced'], 'model__l1_ratio': [0]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'accuracy': 'accuracy', 'f1': make_scorer(f...ro_division=0), 'precision': make_scorer(p...ro_division=0), 'recall': make_scorer(r...ro_division=0), ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo.

In [14]:
print("Best params:")
print(grid_search.best_params_)

print("\nBest CV F1:")
print(grid_search.best_score_)

Best params:
{'model__C': 10, 'model__class_weight': 'balanced', 'model__l1_ratio': 0}

Best CV F1:
0.6820753837698043


In [16]:
grid_results_df = pd.DataFrame(grid_search.cv_results_)

cols = [
    "param_model__C",
    "param_model__l1_ratio",
    "param_model__class_weight",
    "mean_test_accuracy",
    "std_test_accuracy",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_f1",
    "mean_test_roc_auc",
    "mean_train_f1",
]

grid_results_summary = (
    grid_results_df[cols]
    .sort_values("mean_test_f1", ascending=False)
)

grid_results_summary.head(10)

,param_model__C,param_model__l1_ratio,param_model__class_weight,mean_test_accuracy,std_test_accuracy,mean_test_precision,mean_test_recall,mean_test_f1,mean_test_roc_auc,mean_train_f1
4,10.0,0,balanced,0.765467,0.003526,0.648211,0.724474,0.682075,0.843938,0.684471
5,30.0,0,balanced,0.765467,0.003526,0.648211,0.724474,0.682075,0.843822,0.684471
3,3.0,0,balanced,0.763841,0.005200,0.645921,0.724474,0.680645,0.843940,0.684467
1,0.3,0,balanced,0.763841,0.005200,0.645921,0.724474,0.680645,0.844875,0.679070
2,1.0,0,balanced,0.763841,0.005200,0.645921,0.724474,0.680645,0.844114,0.683366
0,0.1,0,balanced,0.757337,0.005708,0.637338,0.715172,0.671729,0.845756,0.676559


In [18]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

best_logreg_pipeline = grid_search.best_estimator_

y_train_pred_oof = cross_val_predict(
    best_logreg_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict",
)

cm = confusion_matrix(y_train, y_train_pred_oof)

print(cm)

print(
    classification_report(
        y_train,
        y_train_pred_oof,
        target_names=["No diabetes", "Diabetes"],
        zero_division=0,
    )
)

[[315  85]
 [ 59 155]]
              precision    recall  f1-score   support

 No diabetes       0.84      0.79      0.81       400
    Diabetes       0.65      0.72      0.68       214

    accuracy                           0.77       614
   macro avg       0.74      0.76      0.75       614
weighted avg       0.77      0.77      0.77       614



In [19]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, roc_auc_score

best_logreg_pipeline = grid_search.best_estimator_

y_train_proba_oof = cross_val_predict(
    best_logreg_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
)[:, 1]

print("OOF ROC-AUC:", roc_auc_score(y_train, y_train_proba_oof))

OOF ROC-AUC: 0.8453504672897196


In [20]:
thresholds = np.arange(0.20, 0.81, 0.05)

threshold_results = []

for threshold in thresholds:
    y_pred_threshold = (y_train_proba_oof >= threshold).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_train, y_pred_threshold).ravel()
    
    threshold_results.append(
        {
            "threshold": threshold,
            "accuracy": accuracy_score(y_train, y_pred_threshold),
            "precision": precision_score(y_train, y_pred_threshold, zero_division=0),
            "recall": recall_score(y_train, y_pred_threshold, zero_division=0),
            "f1": f1_score(y_train, y_pred_threshold, zero_division=0),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
        }
    )

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.sort_values("f1", ascending=False)

,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp
6,0.50,0.765472,0.645833,0.724299,0.682819,315,85,59,155
4,0.40,0.726384,0.575658,0.817757,0.675676,271,129,39,175
8,0.60,0.786645,0.719577,0.635514,0.674938,347,53,78,136
5,0.45,0.742671,0.602941,0.766355,0.674897,292,108,50,164
9,0.65,0.793160,0.754386,0.602804,0.670130,358,42,85,129
7,0.55,0.773616,0.681159,0.658879,0.669834,334,66,73,141
2,0.30,0.679153,0.523035,0.901869,0.662093,224,176,21,193
3,0.35,0.695440,0.540541,0.841121,0.658135,247,153,34,180
1,0.25,0.648208,0.497525,0.939252,0.650485,197,203,13,201
0,0.20,0.615635,0.474771,0.967290,0.636923,171,229,7,207


In [21]:
threshold_results_df.sort_values("recall", ascending=False).head(10)

,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp
0,0.20,0.615635,0.474771,0.967290,0.636923,171,229,7,207
1,0.25,0.648208,0.497525,0.939252,0.650485,197,203,13,201
2,0.30,0.679153,0.523035,0.901869,0.662093,224,176,21,193
3,0.35,0.695440,0.540541,0.841121,0.658135,247,153,34,180
4,0.40,0.726384,0.575658,0.817757,0.675676,271,129,39,175
5,0.45,0.742671,0.602941,0.766355,0.674897,292,108,50,164
6,0.50,0.765472,0.645833,0.724299,0.682819,315,85,59,155
7,0.55,0.773616,0.681159,0.658879,0.669834,334,66,73,141
8,0.60,0.786645,0.719577,0.635514,0.674938,347,53,78,136
9,0.65,0.793160,0.754386,0.602804,0.670130,358,42,85,129


In [22]:
threshold_results_df[
    threshold_results_df["precision"] >= 0.55
].sort_values("recall", ascending=False)

,threshold,accuracy,precision,recall,f1,tn,fp,fn,tp
4,0.40,0.726384,0.575658,0.817757,0.675676,271,129,39,175
5,0.45,0.742671,0.602941,0.766355,0.674897,292,108,50,164
6,0.50,0.765472,0.645833,0.724299,0.682819,315,85,59,155
7,0.55,0.773616,0.681159,0.658879,0.669834,334,66,73,141
8,0.60,0.786645,0.719577,0.635514,0.674938,347,53,78,136
9,0.65,0.793160,0.754386,0.602804,0.670130,358,42,85,129
10,0.70,0.781759,0.770270,0.532710,0.629834,366,34,100,114
11,0.75,0.773616,0.769784,0.500000,0.606232,368,32,107,107
12,0.80,0.768730,0.800000,0.448598,0.574850,376,24,118,96


## Threshold selection

We evaluated decision thresholds using out-of-fold predicted probabilities on X_train.

The default threshold 0.50 gives the best F1:

- precision: 0.646
- recall: 0.724
- F1: 0.683
- FN: 59
- FP: 85

A lower threshold 0.40 gives higher recall:

- precision: 0.576
- recall: 0.818
- F1: 0.676
- FN: 39
- FP: 129

For a screening-like medical task, False Negatives are more dangerous than False Positives.
Therefore, we select threshold = 0.40 as a recall-oriented decision threshold.

The final X_test set is still not used for threshold selection.

## Final evaluation on X_test

The model and threshold were selected using cross-validation on X_train only.

Final configuration:
- LogisticRegression
- C = 10
- class_weight = "balanced"
- l1_ratio = 0
- threshold = 0.40

Now we evaluate this fixed configuration on X_test once.

In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

FINAL_THRESHOLD = 0.40

final_pipeline = grid_search.best_estimator_

# grid_search.best_estimator_ is already refit on X_train,
# but calling fit explicitly makes this final step easy to read.
final_pipeline.fit(X_train, y_train)

y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= FINAL_THRESHOLD).astype(int)

test_cm = confusion_matrix(y_test, y_test_pred)

test_metrics = {
    "accuracy": accuracy_score(y_test, y_test_pred),
    "precision": precision_score(y_test, y_test_pred, zero_division=0),
    "recall": recall_score(y_test, y_test_pred, zero_division=0),
    "f1": f1_score(y_test, y_test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_test_proba),
}

print("Final threshold:", FINAL_THRESHOLD)
print("\nConfusion matrix:")
print(test_cm)

print("\nMetrics:")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        target_names=["No diabetes", "Diabetes"],
        zero_division=0,
    )
)

Final threshold: 0.4

Confusion matrix:
[[68 32]
 [ 7 47]]

Metrics:
accuracy: 0.7468
precision: 0.5949
recall: 0.8704
f1: 0.7068
roc_auc: 0.8165

Classification report:
              precision    recall  f1-score   support

 No diabetes       0.91      0.68      0.78       100
    Diabetes       0.59      0.87      0.71        54

    accuracy                           0.75       154
   macro avg       0.75      0.78      0.74       154
weighted avg       0.80      0.75      0.75       154



Финальная оценка на holdout test подтверждает, что выбранная модель работает в recall-oriented режиме и хорошо находит класс Diabetes, но даёт заметное число False Positive.